In [ ]:
import pandas as pd
import mygene
import numpy as np
import requests
import gzip
import io

# ============================
# 1) ONLINE INTELLIGENCE (FLEXIBLE)
# ============================
print("--- 1. Connecting to the GEO Database for the 11 NORMAL samples ---")
url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE65nnn/GSE65194/matrix/GSE65194_series_matrix.txt.gz"

try:
    response = requests.get(url)
    response.raise_for_status()
    content = gzip.decompress(response.content).decode('utf-8', errors='ignore').split('\n')
    
    geo_ids = []
    descriptions = []
    
    # Robust parsing: search for key rows across the entire file content
    for line in content:
        if "!Sample_geo_accession" in line:
            # Clean unnecessary characters and split text
            geo_ids = [x.strip().strip('"') for x in line.split('\t')[1:]]
        
        # Search within both "source name" and "characteristics" fields
        if "!Sample_source_name_ch1" in line:
            descriptions = [x.strip().strip('"') for x in line.split('\t')[1:]]
            
    # ID-to-Description mapping
    sample_map = dict(zip(geo_ids, descriptions))
    
    # EXTENDED FILTER: Search for any synonym of "Healthy/Normal"
    keywords = ['normal', 'healthy', 'mammoplasty', 'control', 'reduction']
    
    real_normal_ids = []
    print("\n   > Analyzing retrieved samples:")
    
    for gid, desc in sample_map.items():
        desc_lower = desc.lower()
        # Logic: Must contain a "healthy" keyword AND must not be a cell line
        if any(k in desc_lower for k in keywords) and "cell line" not in desc_lower:
            real_normal_ids.append(gid)
            # Print only the first 3 and last 3 entries for visual verification
            if len(real_normal_ids) <= 3 or len(real_normal_ids) > 8:
                print(f"     Identified: {gid} -> {desc}")
    
    print(f"\n   > Total Healthy Samples identified online: {len(real_normal_ids)}")
    
    if len(real_normal_ids) != 11:
        print(f"   ⚠️ WARNING: Found {len(real_normal_ids)} samples instead of 11.")
        print("      Proceeding with the current set of IDs.")
    else:
        print("   ✅ SUCCESS: Exactly 11 healthy samples have been successfully identified.")

except Exception as e:
    print(f"   ❌ CRITICAL ONLINE ERROR: {e}")
    print("   Unable to retrieve IDs. Execution aborted for safety.")
    real_normal_ids = []

# ============================
# 2) LOCAL CSV LOADING
# ============================
if len(real_normal_ids) > 0:
    print("\n--- 2. Cross-referencing with the local CSV file ---")
    expr = pd.read_csv("S1.csv", index_col=0)
    
    # CSV column cleaning (removal of quotes and extra whitespace)
    expr.columns = expr.columns.str.extract(r'(GSM\d+)', expand=False)
    
    # Verify which online-identified Normal samples exist in the local CSV
    final_normal_samples = [s for s in real_normal_ids if s in expr.columns]
    print(f"   > Normal samples present in the local CSV: {len(final_normal_samples)}")
    
    if len(final_normal_samples) == 0:
        print("   ❌ ERROR: None of the IDs identified online match the CSV entries. Please verify column nomenclature.")
    else:
        # ============================
        # 3) DATA EXTRACTION
        # ============================
        print("\n--- 3. Processing and Annotation ---")
        mg = mygene.MyGeneInfo()
        doxo_targets = pd.read_csv("network_genes.csv")
        
        # Annotation process
        probe_ids = expr.index.tolist()
        annotation = mg.querymany(probe_ids, scopes='reporter', fields='symbol', species='human', verbose=False)
        probe_to_gene = {item['query']: item.get('symbol', None) for item in annotation}
        
        expr_annotated = expr.copy()
        expr_annotated["GeneSymbol"] = expr_annotated.index.map(probe_to_gene)
        expr_annotated = expr_annotated.dropna(subset=["GeneSymbol"])
        expr_by_gene = expr_annotated.groupby("GeneSymbol").mean()
        expr_by_gene.index = expr_by_gene.index.str.strip().str.upper()

        # Gene list preparation
        doxo_targets["GeneSymbol"] = doxo_targets["GeneSymbol"].str.strip().str.upper()
        corrections = {"ERK1": "MAPK3", "P21": "CDKN1A", "CYC": "CYCS", "GABP": "GABPA", 
                       "BAK": "BAK1", "DDR1": "DDR1", "EEF1G": "EEF1G"}
        doxo_targets["GeneSymbol"] = doxo_targets["GeneSymbol"].replace(corrections)
        gene_list = doxo_targets["GeneSymbol"].unique()
        
        # Output Generation Function
        def extract_patient_and_normals(patient_id):
            if patient_id not in expr_by_gene.columns:
                print(f"Error: Patient {patient_id} not found.")
                return

            subset = expr_by_gene.loc[expr_by_gene.index.isin(gene_list)].copy()
            
            df_out = pd.DataFrame({"GeneSymbol": subset.index})
            df_out[patient_id] = subset[patient_id].values
            
            for ns in final_normal_samples:
                df_out[ns] = subset[ns].values
                
            filename = f"TNBC_patient_{patient_id}_vs_normals.csv"
            df_out.to_csv(filename, index=False)
            print(f"\n✅ FILE GENERATED: {filename}")
            print(f"   Structure: 1 Patient Column ({patient_id}) + {len(final_normal_samples)} Normal (Healthy) Columns.")

        # EXECUTION (Insert the desired TNBC patient ID below)
        extract_patient_and_normals("GSM158xxxx")
else:
    print("Execution halted: Unable to proceed without valid normal sample IDs.")